In [ ]:
!unzip /content/KinFaceW-I.zip -d /content/

In [ ]:
!unzip /content/KinFaceW-II.zip -d /content/

## 1 · Setup

In [1]:
import os, math, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import scipy.io as sio

# Install facenet_pytorch if not already installed
try:
    import facenet_pytorch
except ImportError:
    !pip install facenet_pytorch

# Face-pretrained backbone
from facenet_pytorch import InceptionResnetV1

SEED = 42
FAST_TRAINING = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    if FAST_TRAINING:
        torch.backends.cudnn.benchmark = True
    else:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print("Imports OK")

Imports OK


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NON_BLOCK = DEVICE.type == "cuda"
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

Device: cuda | AMP: True


## 2 · Config

# Choose a backbone! customCNN, facenet, resnet50, or vgg16.

In [ ]:
BACKBONE   = "simpleCNN"               # "facenet" | "vgg16" | "simpleCNN"
DATASET = "KinFaceW-II"
DRIVE_PATH = f"/content"

In [ ]:
DATA_ROOT = f"{DRIVE_PATH}/{DATASET}"  # SELECT KFW I OR II
IMG_SIZE   = 160                     # 160 for facenet, 112 for resnet50/vgg16
EMBED_DIM  = 512                     # 512 = native FaceNet embedding
BATCH_SIZE = 32
NUM_EPOCHS = 50
NUM_FOLDS  = 5

# Learning rates (separate per component)
LR_ENCODER    = 1e-5                 # Very low — pretrained face features are already good
LR_CLASSIFIER = 5e-4
LR_ARCFACE    = 5e-4
WEIGHT_DECAY  = 1e-4

# Loss weights
W_VERIF     = 1.0                   # BCE verification loss
W_CONTRAST  = 0                    # Contrastive loss (shapes embedding geometry)
W_ARCFACE   = 0                 # ArcFace identity loss

# Early stopping
PATIENCE   = 15
EVAL_EVERY = 1                       # Validate every epoch (dataset is small)

# Gradual unfreezing
FREEZE_ENCODER_EPOCHS = 5            # Freeze backbone for first N epochs

# DataLoader
NUM_WORKERS = 4

RELATIONS = {
    "fd": "father-dau",
    "fs": "father-son",
    "md": "mother-dau",
    "ms": "mother-son",
}
REL_NAMES = {"fd": "Father-Daughter", "fs": "Father-Son",
             "md": "Mother-Daughter", "ms": "Mother-Son"}

# Auto-adjust for different backbones
if BACKBONE in ("resnet50", "vgg16"):
    IMG_SIZE = 112
    EMBED_DIM = 256
    LR_ENCODER = 5e-5

if BACKBONE == "simpleCNN":
  IMG_SIZE = 64
  EMBED_DIM = 256
  LR_ENCODER = .01

print(f"Data root:   {DATA_ROOT}")
print(f"Backbone:    {BACKBONE}")
print(f"Image size:  {IMG_SIZE}x{IMG_SIZE}")
print(f"Embed dim:   {EMBED_DIM}")
print(f"Encoder LR:  {LR_ENCODER}")
print(f"Epochs:      {NUM_EPOCHS}")
print(f"Batch size:  {BATCH_SIZE}")

Data root:   /content/KinFaceW-I
Backbone:    simpleCNN
Image size:  64x64
Embed dim:   256
Encoder LR:  0.01
Epochs:      50
Batch size:  32


Backbone is not defined? Look at the cell before this one.

## 3 · Data Loading

In [221]:
def _extract_str(x):
    """Unwrap nested MATLAB arrays to a plain Python string."""
    while hasattr(x, '__len__') and not isinstance(x, str):
        x = x[0]
    return str(x).strip()


def load_pairs(data_root):
    """Load all kin pairs from .mat metadata files."""
    images_dir = os.path.join(data_root, "images")
    meta_dir   = os.path.join(data_root, "meta_data")
    all_pairs  = []
    missing    = []

    for rel_code, rel_folder in RELATIONS.items():
        mat_path = os.path.join(meta_dir, f"{rel_code}_pairs.mat")
        if not os.path.exists(mat_path):
            print(f"  WARNING: {mat_path} not found")
            continue

        mat = sio.loadmat(mat_path)
        key = next((k for k in mat if "pair" in k.lower()), "pairs")
        raw = mat[key]

        pos, neg = 0, 0
        for row in raw:
            fold  = int(row[0].flat[0]) if hasattr(row[0], 'flat') else int(row[0])
            label = int(row[1].flat[0]) if hasattr(row[1], 'flat') else int(row[1])

            img1 = os.path.join(images_dir, rel_folder, _extract_str(row[2]))
            img2 = os.path.join(images_dir, rel_folder, _extract_str(row[3]))

            for p in [img1, img2]:
                if not os.path.exists(p):
                    missing.append(p)

            all_pairs.append({
                "fold": fold, "label": label,
                "img1": img1, "img2": img2,
                "relation": rel_code,
            })
            if label == 1: pos += 1
            else: neg += 1

        print(f"  {rel_code}: {pos} pos + {neg} neg = {pos+neg}")

    print(f"  Total: {len(all_pairs)} pairs")

    if missing:
        unique_missing = list(set(missing))
        print(f"\n  {len(unique_missing)} image files not found!")
        for p in unique_missing[:5]:
            print(f"    {p}")

    return all_pairs


pairs = load_pairs(DATA_ROOT)
if not pairs:
    raise RuntimeError(
        f"No pairs loaded from DATA_ROOT={DATA_ROOT!r}. "
        "Download KinFaceW-I/II, unzip so meta_data contains fd_pairs.mat etc., "
        "or set DATA_ROOT to the folder that contains images/ and meta_data/."
    )

# Identity map for ArcFace
all_images = sorted(set(img for p in pairs for img in [p["img1"], p["img2"]]))
img_to_id = {img: i for i, img in enumerate(all_images)}
NUM_IDS = len(img_to_id)
print(f"Unique faces: {NUM_IDS}")

  fd: 134 pos + 134 neg = 268
  fs: 156 pos + 156 neg = 312
  md: 127 pos + 127 neg = 254
  ms: 116 pos + 116 neg = 232
  Total: 1066 pairs
Unique faces: 1066


## 4 · Dataset & Transforms

In [ ]:
# Define normalization stats per backbone
normalize_mean_and_std = {
    "vgg16" : {"mean" : [0.485, 0.456, 0.406],
               "std" : [0.229, 0.224, 0.225]},
    "facenet" : {"mean" : [0.5] * 3,
               "std" : [0.5] * 3},
    "simpleCNN" : {"mean" : 0,
               "std" : 1}
}

major_transform = T.Compose([
    T.Resize((IMG_SIZE + 8, IMG_SIZE + 8)),
    T.RandomHorizontalFlip(),                                               # Add random horizontal flip to increase variation (e.g. left-right head pose)
    T.RandomRotation(15),                                                   # Add random rotation to imitate different head poses
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),   # Add jitter to imitate different lighting conditions and color variations
    T.RandomGrayscale(p=0.1),                                               # Randomly convert some images to grayscale to make model focus on features, not just color
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),                        # Add slight blur to imitate blurry photos
    T.ToTensor(),
    T.Normalize(mean=normalize_mean_and_std[BACKBONE]["mean"],
                std=normalize_mean_and_std[BACKBONE]["std"]),
])

minor_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=normalize_mean_and_std[BACKBONE]["mean"],
                std=normalize_mean_and_std[BACKBONE]["std"]),
])

resizing_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=normalize_mean_and_std[BACKBONE]["mean"],
                std=normalize_mean_and_std[BACKBONE]["std"])
])

simple_cnn_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

if BACKBONE != "simpleCNN":
    val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=normalize_mean_and_std[BACKBONE]["mean"],
                std=normalize_mean_and_std[BACKBONE]["std"])
    ])
    
    test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=normalize_mean_and_std[BACKBONE]["mean"],
                std=normalize_mean_and_std[BACKBONE]["std"])
    ])
else:
    val_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(), # This will convert pixel values to [0, 1], which is important for simpleCNN
    ])

    test_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(), # This will convert pixel values to [0, 1], which is important for simpleCNN
    ])


class KinDataset(Dataset):
    """Returns (img1, img2, label, id1, id2, relation_idx)."""

    REL_IDX = {"fd": 0, "fs": 1, "md": 2, "ms": 3}

    def __init__(self, pairs, img_to_id, transform):
        self.pairs = pairs
        self.img_to_id = img_to_id
        self.tf = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        pil1 = Image.open(p["img1"]).convert("RGB")
        pil2 = Image.open(p["img2"]).convert("RGB")
        return (
            self.tf(pil1), self.tf(pil2),
            torch.tensor(p["label"], dtype=torch.float32),
            torch.tensor(self.img_to_id[p["img1"]], dtype=torch.long),
            torch.tensor(self.img_to_id[p["img2"]], dtype=torch.long),
            torch.tensor(self.REL_IDX[p["relation"]], dtype=torch.long),
        )


print("Dataset ready.")

Dataset ready.


## 5 · Model
### 5.1 Encoder Backbones

In [ ]:
class FaceNetBackbone(nn.Module):
    """
    InceptionResnetV1 pretrained on VGGFace2 (3.3M face images, 9131 identities).
    This is the KEY improvement — face-specific features instead of ImageNet features.

    Input:  (B, 3, 160, 160)
    Output: (B, 512) normalized embeddings
    """

    def __init__(self, embed_dim=512):
        super().__init__()
        self.facenet = InceptionResnetV1(pretrained='vggface2')

        # Replace FaceNet's final linear if we want different embed_dim
        if embed_dim != 512:
            self.proj = nn.Sequential(
                nn.Linear(512, embed_dim),
                nn.BatchNorm1d(embed_dim),
                nn.Dropout(0.3),
            )
        else:
            self.proj = nn.Sequential(
                nn.BatchNorm1d(512),
                nn.Dropout(0.3),
            )

    def forward(self, x):
        features = self.facenet(x)
        return self.proj(features)



class VGG16Backbone(nn.Module):
    """VGG16 adapted for 112x112 input."""

    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        vgg = models.vgg16_bn(weights="IMAGENET1K_V1" if pretrained else None)
        self.features = vgg.features[:34]
        self.features_tail = vgg.features[34:43]
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Linear(512, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.features_tail(x)
        x = self.pool(x).flatten(1)
        return self.proj(x)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: learns to weight channels by importance."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

class ResidualBlock(nn.Module):
    """Conv-BN-ReLU-Conv-BN + residual skip. Optional SE attention."""

    def __init__(self, in_ch, out_ch, stride=1, use_se=True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock(out_ch) if use_se else nn.Identity()

        # 1x1 projection for dimension/stride mismatch
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.se(self.conv(x)) + self.skip(x))

class SimpleCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(64, 1024)
        self.fc2 = nn.Linear(1024, embed_dim)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Factory
BACKBONES = {
    "facenet":    FaceNetBackbone,
    "vgg16":      VGG16Backbone,
    "simpleCNN": SimpleCNN
}

def get_backbone(name, embed_dim):
    if name not in BACKBONES:
        raise ValueError(f"Unknown backbone '{name}'. Choose from: {list(BACKBONES.keys())}")
    return BACKBONES[name](embed_dim)


# Test
print(f"Testing backbones with {IMG_SIZE}x{IMG_SIZE} input:\n")
for name in BACKBONES:
    sz = 160 if name == "facenet" else 112
    m = get_backbone(name, 512 if name == "facenet" else 256)
    x = torch.randn(2, 3, sz, sz)
    o = m(x)
    p = sum(x.numel() for x in m.parameters())
    print(f"  {name:12s} -> {tuple(o.shape)}, params: {p:>12,}")
    del m, x, o

Testing backbones with 64x64 input:

  facenet      -> (2, 512), params:   27,911,351
  resnet50     -> (2, 256), params:   24,025,408
  vgg16        -> (2, 256), params:   14,854,976
  customCNN    -> (2, 256), params:   11,387,712
  simpleCNN    -> (2, 256), params:      348,352


### 5.2 Contrastive Loss

In [224]:
class ContrastiveLoss(nn.Module):
    """
    Contrastive loss (Chopra et al., 2005).
    Pulls kin-pairs together, pushes non-kin apart beyond margin.
    Directly shapes the embedding space geometry.
    """

    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        dist = F.pairwise_distance(emb1, emb2)
        # labels=1 means kin (pull together), labels=0 means non-kin (push apart)
        loss = labels * dist.pow(2) + (1 - labels) * F.relu(self.margin - dist).pow(2)
        return loss.mean() * 0.5


print("ContrastiveLoss ready.")

ContrastiveLoss ready.


### 5.3 ArcFace Loss

In [225]:
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss (Deng et al., CVPR 2019)."""

    def __init__(self, num_classes, embed_dim=512, s=30.0, m=0.3):
        super().__init__()
        self.s, self.m = s, m
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.threshold = math.cos(math.pi - m)
        self.W = nn.Parameter(torch.randn(num_classes, embed_dim))
        nn.init.xavier_normal_(self.W)

    def forward(self, embeddings, labels):
        emb = F.normalize(embeddings, p=2, dim=1)
        W   = F.normalize(self.W, p=2, dim=1)
        cos = F.linear(emb, W).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1.0 - cos ** 2)
        cos_m = cos * self.cos_m - sin * self.sin_m
        cos_m = torch.where(cos > self.threshold, cos_m, cos - self.sin_m * self.m)
        one_hot = F.one_hot(labels, self.W.size(0)).float()
        logits = (one_hot * cos_m + (1 - one_hot) * cos) * self.s
        return F.cross_entropy(logits, labels)

### 5.4 Verification Head (attention-weighted fusion)

In [226]:
class VerificationHead(nn.Module):
    """
    Fuses two embeddings -> kin/non-kin prediction.

    v5 improvements:
    - Added cosine similarity as explicit scalar feature
    - Attention gate to weight feature dimensions
    - Larger first hidden layer for richer features
    """

    def __init__(self, embed_dim=512):
        super().__init__()
        feat_dim = embed_dim * 3 + 1   # |diff|, product, sq_diff, cosine_sim

        # Attention gate — learns which feature dimensions matter most
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 4),
            nn.ReLU(),
            nn.Linear(feat_dim // 4, feat_dim),
            nn.Sigmoid(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, emb1, emb2):
        diff    = torch.abs(emb1 - emb2)
        prod    = emb1 * emb2
        # ERICK FIX 1
        sq_diff = (emb1 - emb2) ** 2
        cos_sim = F.cosine_similarity(emb1, emb2, dim=1).unsqueeze(1)

        fused = torch.cat([diff, prod, sq_diff, cos_sim], dim=1)
        att   = self.attention(fused)
        fused = fused * att
        return self.classifier(fused)

### 5.5 Siamese Network



In [ ]:
class KinshipNet(nn.Module):
    """
    Siamese network for kinship verification.
    """

    def __init__(self, backbone_name, embed_dim, num_ids):
        super().__init__()
        self.encoder  = get_backbone(backbone_name, embed_dim)
        self.verifier = VerificationHead(embed_dim)
        self.arcface  = ArcFaceLoss(num_ids, embed_dim, s=30.0, m=0.3)
        self.contrast = ContrastiveLoss(margin=2.0)

    def encode(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1) # L2 normalize embeddings to lie on unit hypersphere, dim=1 means normalize the vector for each sample in the batch

    def forward(self, img1, img2):
        emb1 = self.encode(img1) # L2 normalized embedding
        emb2 = self.encode(img2) # L2 normalized embedding
        logits = self.verifier(emb1, emb2) # raw logits for BCEWithLogitsLoss
        return logits, emb1, emb2

    def freeze_encoder(self):
        """Freeze backbone weights — train only heads."""
        for param in self.encoder.parameters():
            param.requires_grad = False

    def unfreeze_encoder(self):
        """Unfreeze backbone for fine-tuning."""
        for param in self.encoder.parameters():
            param.requires_grad = True


# Build & verify
model = KinshipNet(BACKBONE, EMBED_DIM, NUM_IDS).to(DEVICE)
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_l, _e1, _e2 = model(_x, _x)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"KinshipNet [{BACKBONE}]")
print(f"  Embedding: {tuple(_e1.shape)}, Logits: {tuple(_l.shape)}")
print(f"  Total params: {total:,}, Trainable: {trainable:,}")
del _x, _l, _e1, _e2, model

KinshipNet [simpleCNN]
  Embedding: (2, 256), Logits: (2, 1)
  Total params: 1,378,818, Trainable: 1,378,818


## 6 · Training Engine

In [ ]:
def train_epoch(model, loader, optimizer, scaler, epoch):
    """Train one epoch with BCE + Contrastive + ArcFace."""
    model.train()
    bce_fn = nn.BCEWithLogitsLoss()
    totals = defaultdict(float)
    n = 0

    for batch in loader:
        img1, img2, labels, id1, id2, rel = [
            x.to(DEVICE, non_blocking=NON_BLOCK) for x in batch
        ]

        with autocast(enabled=USE_AMP):
            logits, emb1, emb2 = model(img1, img2)

            loss_v = bce_fn(logits.squeeze(1), labels)
            loss_c = model.contrast(emb1, emb2, labels)
            loss_a = (model.arcface(emb1, id1) + model.arcface(emb2, id2)) / 2

            loss = W_VERIF * loss_v + W_CONTRAST * loss_c + W_ARCFACE * loss_a

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        bs = img1.size(0)
        totals["loss"] += loss.item() * bs
        totals["v"]    += loss_v.item() * bs
        totals["c"]    += loss_c.item() * bs
        totals["a"]    += loss_a.item() * bs

        # ERICK FIX 2
        # training accuracy
        preds = (torch.sigmoid(logits.squeeze(1)) >= 0.5).float()
        totals["correct"] += (preds == labels).sum().item()
        n += bs

    res = {k: v / n for k, v in totals.items() if k != "correct"}
    res["acc"] = totals["correct"] / n
    return res


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate -> dict with acc, auc, per-relation breakdown, and loss."""
    model.eval()
    bce_fn = nn.BCEWithLogitsLoss() # NEW: Loss function
    all_labels, all_probs, all_rels = [], [], []
    total_loss = 0.0 # NEW
    n = 0 # NEW

    for batch in loader:
        img1, img2, labels, id1, id2, rels = [ # Added id1, id2 for ArcFace loss if needed
            x.to(DEVICE, non_blocking=NON_BLOCK) for x in batch
        ]

        with autocast(enabled=USE_AMP):
            logits, emb1, emb2 = model(img1, img2)

            # NEW: Calculate validation loss (consistent with training)
            loss_v = bce_fn(logits.squeeze(1), labels)
            # If you want validation loss to match training loss (including Contrastive/ArcFace):
            loss_c = model.contrast(emb1, emb2, labels)
            loss_a = (model.arcface(emb1, id1) + model.arcface(emb2, id2)) / 2
            batch_loss = W_VERIF * loss_v + W_CONTRAST * loss_c + W_ARCFACE * loss_a

        bs = img1.size(0)
        total_loss += batch_loss.item() * bs # NEW
        n += bs # NEW

        probs = torch.sigmoid(logits.squeeze(1))
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_rels.extend(rels.cpu().numpy())

    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    rels   = np.array(all_rels)
    preds  = (probs >= 0.6).astype(int)

    result = {
        "loss": total_loss / n, # NEW: Average validation loss
        "acc": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
        "labels": labels, "probs": probs,
    }

    # ... (rest of your per-relation code remains the same)
    rel_idx = {"fd": 0, "fs": 1, "md": 2, "ms": 3}
    for rel_code, idx in rel_idx.items():
        mask = (rels == idx)
        if mask.sum() > 0 and len(np.unique(labels[mask])) > 1:
            result[f"acc_{rel_code}"] = accuracy_score(labels[mask], preds[mask])
            result[f"auc_{rel_code}"] = roc_auc_score(labels[mask], probs[mask])

    return result

print("Training engine ready.")

Training engine ready.


## 7 · 5-Fold Cross-Validation

#### Will change so each fold has 50/50 pos/neg pairs, and we can do stratified sampling to maintain balance in train/val splits.

In [ ]:
# fold_results = []
# best_model_state = None
# best_global_auc = 0

# for fold in range(1, NUM_FOLDS + 1):
#     print(f"\n{'='*62}")
#     print(f"  FOLD {fold}/{NUM_FOLDS}  |  {BACKBONE}  |  {IMG_SIZE}x{IMG_SIZE}")
#     print(f"{'='*62}")
#     # ERICK FIX 3
#     raw_train_p = [p for p in pairs if p["fold"] != fold]
#     test_p  = [p for p in pairs if p["fold"] == fold]

#     # Shuffle and split 10% of the training data for validation
#     random.shuffle(raw_train_p)
#     split_idx = int(len(raw_train_p) * 0.9)
#     train_p = raw_train_p[:split_idx]
#     val_p   = raw_train_p[split_idx:]

#     _pin = DEVICE.type == "cuda"
#     _pw = NUM_WORKERS > 0
#     train_dl = DataLoader(
#         KinDataset(train_p, img_to_id, train_tf4),
#         batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
#         drop_last=True, pin_memory=_pin, persistent_workers=_pw,
#     )
#     val_dl = DataLoader(
#         KinDataset(val_p, img_to_id, val_tf),
#         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
#         pin_memory=_pin, persistent_workers=_pw,
#     )
#     test_dl = DataLoader(
#         KinDataset(test_p, img_to_id, val_tf),
#         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
#         pin_memory=_pin, persistent_workers=_pw,
#     )
#     # hi

#     # Fresh model
#     model = KinshipNet(BACKBONE, EMBED_DIM, NUM_IDS).to(DEVICE)

#     # ── Gradual unfreezing ──
#     # Phase 1: Freeze encoder, train only heads (FREEZE_ENCODER_EPOCHS epochs)
#     # Phase 2: Unfreeze encoder with low LR for fine-tuning
#     model.freeze_encoder()

#     # Phase 1 optimizer — heads only
#     head_params = [
#         {"params": model.verifier.parameters(), "lr": LR_CLASSIFIER},
#         {"params": model.arcface.parameters(),  "lr": LR_ARCFACE},
#     ]
#     optimizer = torch.optim.AdamW(head_params, weight_decay=WEIGHT_DECAY)
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#         optimizer, T_max=FREEZE_ENCODER_EPOCHS, eta_min=1e-6,
#     )
#     scaler = GradScaler(enabled=USE_AMP)

#     # Tracking
#     best_auc, best_acc = 0, 0
#     best_fold_state = None
#     patience_counter = 0
#     history = defaultdict(list)
#     unfrozen = False

#     t0 = time.time()
#     for ep in range(NUM_EPOCHS):
#         # Unfreeze encoder after warmup phase
#         if ep == FREEZE_ENCODER_EPOCHS and not unfrozen:
#             model.unfreeze_encoder()
#             unfrozen = True
#             print(f"  >> Encoder unfrozen at epoch {ep+1}")

#             # Rebuild optimizer with all params (halve head LR for phase 2)
#             all_params = [
#                 {"params": model.encoder.parameters(),  "lr": LR_ENCODER},
#                 {"params": model.verifier.parameters(), "lr": LR_CLASSIFIER * 0.5},
#                 {"params": model.arcface.parameters(),  "lr": LR_ARCFACE * 0.5},
#             ]
#             optimizer = torch.optim.AdamW(all_params, weight_decay=WEIGHT_DECAY)
#             remaining_epochs = NUM_EPOCHS - ep
#             scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#                 optimizer, T_max=remaining_epochs, eta_min=1e-7,
#             )
#             # scaler = GradScaler(enabled=USE_AMP)

#         losses = train_epoch(model, train_dl, optimizer, scaler, ep)
#         scheduler.step()

#         # Validate
#         if ep % EVAL_EVERY == 0 or ep == NUM_EPOCHS - 1:
#             # ERICK FIX
#             # USE VAL DL
#             res = evaluate(model, val_dl)
#             history["train_loss"].append(losses["loss"])
#             history["valid_loss"].append(res["loss"])
#             history["epoch"].append(ep + 1)
#             history["train_acc"].append(losses["acc"]) # training accuracy
#             history["acc"].append(res["acc"]) # validation accuracy
#             history["auc"].append(res["auc"]) # validation auc

#             rel_str = "  ".join(
#                 f"{r}={res.get(f'acc_{r}', 0):.3f}"
#                 for r in ["fs", "fd", "ms", "md"]
#             )
#             frozen_str = "FROZEN" if not unfrozen else ""
#             print(f"  Ep {ep+1:2d} | L={losses['loss']:.3f} "
#                   f"(v={losses['v']:.3f} c={losses['c']:.3f} a={losses['a']:.3f}) "
#                   f"| Acc={res['acc']:.4f} AUC={res['auc']:.4f} | {rel_str} {frozen_str}")
#             if res["auc"] > best_auc:
#                 best_auc, best_acc = res["auc"], res["acc"]
#                 best_fold_state = copy.deepcopy(model.state_dict())
#                 patience_counter = 0
#             else:
#                 patience_counter += 1

#             if patience_counter >= PATIENCE:
#                 print(f"  Early stop at epoch {ep+1}")
#                 break

#     model.load_state_dict(best_fold_state)
#     final_test_res = evaluate(model, test_dl)
#     test_acc = final_test_res["acc"]
#     test_auc = final_test_res["auc"]

#     elapsed = time.time() - t0
#     print(f"  Fold {fold} Test: Acc={test_acc:.4f} | AUC={test_auc:.4f} | Time={elapsed:.0f}s")
#     print(f"  Fold {fold} Best Val AUC: {best_auc:.4f}")

#     # Use validation AUC to select the best global model, avoiding test leakage
#     if best_auc > best_global_auc:
#         best_global_auc = best_auc
#         best_model_state = best_fold_state

#     fold_results.append({
#         "fold": fold, "best_acc": test_acc, "best_auc": test_auc, # Key names remain the same so summary block works
#         "results": final_test_res,
#         "history": dict(history),
#     })

#     del model, optimizer, scheduler, scaler
#     torch.cuda.empty_cache()


# # Summary
# accs = [r["best_acc"] for r in fold_results]
# aucs = [r["best_auc"] for r in fold_results]
# tta_accs = [r["tta_results"]["acc"] for r in fold_results]

# print(f"\n{'='*62}")
# print(f"  RESULTS: {BACKBONE} {IMG_SIZE}x{IMG_SIZE}")
# print(f"{'='*62}")
# for r in fold_results:
#     print(f"  Fold {r['fold']}: Acc={r['best_acc']:.4f}  AUC={r['best_auc']:.4f}  TTA_Acc={r['tta_results']['acc']:.4f}")
# print(f"\n  Mean Acc:     {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
# print(f"  Mean AUC:     {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}")
# print(f"  Mean TTA Acc: {np.mean(tta_accs):.4f} +/- {np.std(tta_accs):.4f}")

# print(f"\n  Per-relation accuracy:")
# for rel in ["fs", "fd", "ms", "md"]:
#     vals = [r["results"].get(f"acc_{rel}", None) for r in fold_results]
#     vals = [v for v in vals if v is not None]
#     if vals:
#         print(f"    {REL_NAMES[rel]:20s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")


  FOLD 1/5  |  simpleCNN  |  64x64
  Ep  1 | L=0.648 (v=0.648 c=0.894 a=17.406) | Acc=0.4651 AUC=0.6948 | fs=0.550  fd=0.500  ms=0.333  md=0.462 FROZEN
  Ep  2 | L=0.595 (v=0.595 c=0.894 a=17.406) | Acc=0.4651 AUC=0.8065 | fs=0.550  fd=0.500  ms=0.333  md=0.462 FROZEN
  Ep  3 | L=0.601 (v=0.601 c=0.894 a=17.406) | Acc=0.4651 AUC=0.7894 | fs=0.550  fd=0.500  ms=0.333  md=0.462 FROZEN
  Ep  4 | L=0.581 (v=0.581 c=0.894 a=17.406) | Acc=0.5814 AUC=0.8310 | fs=0.700  fd=0.500  ms=0.556  md=0.577 FROZEN
  Ep  5 | L=0.577 (v=0.577 c=0.894 a=17.406) | Acc=0.6977 AUC=0.7880 | fs=0.550  fd=0.636  ms=0.833  md=0.769 FROZEN
  >> Encoder unfrozen at epoch 6
  Ep  6 | L=0.649 (v=0.649 c=0.974 a=17.471) | Acc=0.5814 AUC=0.7571 | fs=0.500  fd=0.545  ms=0.722  md=0.577 
  Ep  7 | L=0.643 (v=0.643 c=0.975 a=17.475) | Acc=0.7326 AUC=0.7804 | fs=0.550  fd=0.773  ms=0.889  md=0.731 
  Ep  8 | L=0.607 (v=0.607 c=0.987 a=17.397) | Acc=0.4651 AUC=0.6959 | fs=0.550  fd=0.500  ms=0.333  md=0.462 
  Ep  9 | L=0

## 8 · Visualization

In [ ]:
# 1. ROC curves
for r in fold_results:
    res = r["results"]
    fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
    auc_val = roc_auc_score(res["labels"], res["probs"])
    plt.plot(fpr, tpr, label=f"Fold {r['fold']} ({auc_val:.3f})", alpha=0.8)
plt.plot([0,1], [0,1], 'k--', alpha=0.3)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title(f"ROC: {BACKBONE}")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 2. Accuracy bars
plt.bar(range(1, NUM_FOLDS+1), accs, color='steelblue', alpha=0.8)
plt.axhline(np.mean(accs), color='red', ls='--', label=f"Mean={np.mean(accs):.3f}")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("Per-fold accuracy")
plt.ylim((0.5, 1.0))
plt.legend()
plt.show()

In [ ]:
# 3. Learning curves
for r in fold_results:
    h = r["history"]
    plt.plot(h["epoch"], h["auc"], alpha=0.7, label=f"Fold {r['fold']}")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Validation AUC over training")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 4. Train vs Validation Accuracy
max_epochs_run = max(len(r["history"]["epoch"]) for r in fold_results)
avg_train_acc = np.zeros(max_epochs_run)
avg_val_acc = np.zeros(max_epochs_run)
counts = np.zeros(max_epochs_run)

for r in fold_results:
    h = r["history"]
    for i, epoch_idx in enumerate(h["epoch"]):
        # h["epoch"] stores 1-based epoch numbers, so we use i for indexing
        avg_train_acc[i] += h["train_acc"][i]
        avg_val_acc[i] += h["acc"][i]
        counts[i] += 1

# Avoid division by zero for epochs where some folds early-stopped
valid_epochs = counts > 0
avg_train_acc[valid_epochs] /= counts[valid_epochs]
avg_val_acc[valid_epochs] /= counts[valid_epochs]
epochs_x = np.arange(1, max_epochs_run + 1)

plt.plot(epochs_x[valid_epochs], avg_train_acc[valid_epochs], label="Train Acc", color='blue', lw=2)
plt.plot(epochs_x[valid_epochs], avg_val_acc[valid_epochs], label="Val Acc", color='orange', lw=2)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Val Accuracy (Mean)")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 4. Train vs Validation Loss
max_epochs_run_loss = max(len(r["history"]["epoch"]) for r in fold_results)
avg_train_loss = np.zeros(max_epochs_run_loss)
avg_val_loss = np.zeros(max_epochs_run_loss)
counts_loss = np.zeros(max_epochs_run_loss)

for r in fold_results:
    h = r["history"]
    for i, epoch_idx in enumerate(h["epoch"]):
        # h["epoch"] stores 1-based epoch numbers, so we use i for indexing
        avg_train_loss[i] += h["train_loss"][i]
        avg_val_loss[i] += h["valid_loss"][i]
        counts_loss[i] += 1

# Avoid division by zero for epochs where some folds early-stopped
valid_epochs_loss = counts_loss > 0
avg_train_loss[valid_epochs_loss] /= counts_loss[valid_epochs_loss]
avg_val_loss[valid_epochs_loss] /= counts_loss[valid_epochs_loss]
epochs_x_loss = np.arange(1, max_epochs_run_loss + 1)

plt.plot(epochs_x_loss[valid_epochs_loss], avg_train_loss[valid_epochs_loss], label="Train Loss", color='blue', lw=2)
plt.plot(epochs_x_loss[valid_epochs_loss], avg_val_loss[valid_epochs_loss], label="Val Loss", color='orange', lw=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss (Mean)")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Per-relation accuracy heatmap
rel_codes = ["fs", "fd", "ms", "md"]
rel_labels = [REL_NAMES[r] for r in rel_codes]

data = np.zeros((NUM_FOLDS, 4))
for i, r in enumerate(fold_results):
    for j, rel in enumerate(rel_codes):
        data[i, j] = r["results"].get(f"acc_{rel}", 0.5)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(data, cmap="RdYlGn", vmin=0.4, vmax=1.0, aspect="auto")
ax.set_xticks(range(4)); ax.set_xticklabels(rel_labels, fontsize=10)
ax.set_yticks(range(NUM_FOLDS)); ax.set_yticklabels([f"Fold {i+1}" for i in range(NUM_FOLDS)])

for i in range(NUM_FOLDS):
    for j in range(4):
        ax.text(j, i, f"{data[i,j]:.2f}", ha="center", va="center", fontsize=10)

plt.colorbar(im, ax=ax, label="Accuracy")
plt.title(f"Per-relation Accuracy: {BACKBONE}")
plt.tight_layout(); plt.show()

## 9 · Save & Load

In [ ]:
import os
import torch
import numpy as np

# 1. Setup paths
SAVED_MODELS_DIR = f"{DRIVE_PATH}/saved_models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

base_filename = f"{DATASET}_{BACKBONE}"
extension = ".pth"
SAVE_PATH = os.path.join(SAVED_MODELS_DIR, f"{base_filename}{extension}")

# 2. Versioning Logic: Check if file exists and increment suffix
counter = 2
while os.path.exists(SAVE_PATH):
    SAVE_PATH = os.path.join(SAVED_MODELS_DIR, f"{base_filename}_{counter}{extension}")
    counter += 1

# 3. Save the file
torch.save({
    "model_state": best_model_state,
    "backbone": BACKBONE,
    "embed_dim": EMBED_DIM,
    "img_size": IMG_SIZE,
    "num_ids": NUM_IDS,
    "mean_acc": np.mean(accs),
    "mean_auc": np.mean(aucs),
    "data_root": DATA_ROOT,
    "accs": accs,
    "fold_results": [{"fold": r["fold"], "acc": r["best_acc"], "auc": r["best_auc"], "history": r["history"], "results": r["results"]}
                     for r in fold_results],
}, SAVE_PATH)

print(f"Saved to {SAVE_PATH} ({os.path.getsize(SAVE_PATH)/1e6:.1f} MB)")

In [ ]:
# Load in a new session (run cells 1-5 first)
# erick fix
MODEL_PATH = SAVE_PATH # Change this

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model = KinshipNet(ckpt["backbone"], ckpt["embed_dim"], ckpt["num_ids"]).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded {ckpt['backbone']} (acc={ckpt['mean_acc']:.4f}, auc={ckpt['mean_auc']:.4f})")

## 10 · Predict

In [ ]:
@torch.no_grad()
def predict(model, path1, path2):
    """Predict kinship between two face images."""
    model.eval()
    img1 = val_tf(Image.open(path1).convert("RGB")).unsqueeze(0).to(DEVICE)
    img2 = val_tf(Image.open(path2).convert("RGB")).unsqueeze(0).to(DEVICE)

    logits, emb1, emb2 = model(img1, img2)
    prob = torch.sigmoid(logits).item()
    cos = F.cosine_similarity(emb1, emb2).item()

    # TTA: also try flipped
    logits_f, _, _ = model(torch.flip(img1, [3]), torch.flip(img2, [3]))
    prob_tta = (prob + torch.sigmoid(logits_f).item()) / 2

    print(f"Kin probability:       {prob:.4f}")
    print(f"Kin probability (TTA): {prob_tta:.4f}")
    print(f"Cosine similarity:     {cos:.4f}")
    print(f"Verdict: {'KIN' if prob_tta >= 0.5 else 'NOT KIN'}")
    return prob_tta, cos


# Example:
# predict(model, "path/to/face1.jpg", "path/to/face2.jpg")

In [ ]:
# ── Batch prediction on sampled pairs → collect results for charts ────────────
# Replaces the noisy per-pair print loop with a clean batch that feeds
# directly into the visualizations below.

from sklearn.metrics import f1_score

DATASET = "KinFaceW-I"
DATA_ROOT = f"{DRIVE_PATH}/{DATASET}"
pairs = load_pairs(DATA_ROOT)

if "pairs" not in globals():
    raise RuntimeError("Run data-loading cells first so 'pairs' is available.")

NUM_SAMPLES = min(200, len(pairs))
random.seed(SEED)
sampled = random.sample(pairs, NUM_SAMPLES)

pred_probs_list, true_labels_list, cos_sims_list, rel_idx_list = [], [], [], []

REL_IDX_MAP = {"fd": 0, "fs": 1, "md": 2, "ms": 3}

print(f"Running batch prediction on {NUM_SAMPLES} sampled pairs...")
model.eval()
with torch.no_grad():
    for p in sampled:
        if not (os.path.exists(p["img1"]) and os.path.exists(p["img2"])):
            continue
        img1 = val_tf(Image.open(p["img1"]).convert("RGB")).unsqueeze(0).to(DEVICE)
        img2 = val_tf(Image.open(p["img2"]).convert("RGB")).unsqueeze(0).to(DEVICE)

        logits, emb1, emb2 = model(img1, img2)
        # TTA: average with flipped prediction
        logits_f, _, _ = model(torch.flip(img1, [3]), torch.flip(img2, [3]))
        prob = (torch.sigmoid(logits.squeeze()).item() +
                torch.sigmoid(logits_f.squeeze()).item()) / 2
        cos  = F.cosine_similarity(emb1, emb2).item()

        pred_probs_list.append(prob)
        true_labels_list.append(p["label"])
        cos_sims_list.append(cos)
        rel_idx_list.append(REL_IDX_MAP[p["relation"]])

# Convert to numpy for downstream charts
pred_probs_arr  = np.array(pred_probs_list)
true_labels_arr = np.array(true_labels_list)
pred_labels_arr = (pred_probs_arr >= 0.5).astype(int)
cos_sims_arr    = np.array(cos_sims_list)
rel_idx_arr     = np.array(rel_idx_list)

acc_batch = accuracy_score(true_labels_arr, pred_labels_arr)
auc_batch = roc_auc_score(true_labels_arr, pred_probs_arr)
f1_batch  = f1_score(true_labels_arr, pred_labels_arr)
print(f"Batch ({NUM_SAMPLES} pairs) — Acc={acc_batch:.4f} | AUC={auc_batch:.4f} | F1={f1_batch:.4f}")
print("Results stored in pred_probs_arr / true_labels_arr — charts follow.")


In [ ]:
# ── Aggregate test metrics across folds ────────────────────────────────────────
print("\n--- Test Metrics Across Folds (held-out test data) ---\n")

all_fprs, all_accs = [], []
all_tps, all_tns, all_fps, all_fns = [], [], [], []

for r in fold_results:
    lbl   = r["results"]["labels"]
    probs = r["results"]["probs"]
    prd   = (probs >= 0.5).astype(int)
    tp = int(((prd == 1) & (lbl == 1)).sum())
    tn = int(((prd == 0) & (lbl == 0)).sum())
    fp = int(((prd == 1) & (lbl == 0)).sum())
    fn = int(((prd == 0) & (lbl == 1)).sum())

    fold_acc = (tp + tn) / (tp + tn + fp + fn)
    fold_fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    all_tps.append(tp); all_tns.append(tn)
    all_fps.append(fp); all_fns.append(fn)
    all_accs.append(fold_acc); all_fprs.append(fold_fpr)
    print(f"  Fold {r['fold']}: Acc={fold_acc:.4f}  FPR={fold_fpr:.4f}  "
          f"TP={tp} TN={tn} FP={fp} FN={fn}")

accuracy           = np.mean(all_accs)
false_positive_rate = np.mean(all_fprs)
print(f"\n  Mean Test Accuracy: {accuracy:.4f} +/- {np.std(all_accs):.4f}")
print(f"  Mean Test FPR:      {false_positive_rate:.4f} +/- {np.std(all_fprs):.4f}")


In [ ]:
# ── Chart 1: Score Distribution (Kin vs Non-Kin) ───────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import ( precision_recall_curve, average_precision_score,
                              f1_score, roc_curve, roc_auc_score)

kin_probs     = pred_probs_arr[true_labels_arr == 1]
non_kin_probs = pred_probs_arr[true_labels_arr == 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(non_kin_probs, bins=30, alpha=0.65, color="salmon",    label="Non-Kin")
ax.hist(kin_probs,     bins=30, alpha=0.65, color="steelblue", label="Kin")
ax.axvline(0.6, color="black", linestyle="--", linewidth=1.5, label="Threshold = 0.6")
ax.set_xlabel("Predicted Kin Probability"); ax.set_ylabel("Count")
ax.set_title(f"Score Distribution — {BACKBONE}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Chart 4: Per-Relation Accuracy Bar (mean ± std across folds) ─────────────
rel_codes  = ["fd", "fs", "md", "ms"]
rel_labels_display = [REL_NAMES[r] for r in rel_codes]
rel_means, rel_stds = [], []

for rel in rel_codes:
    vals = [r["results"].get(f"acc_{rel}") for r in fold_results]
    vals = [v for v in vals if v is not None]
    rel_means.append(np.mean(vals) if vals else 0)
    rel_stds.append(np.std(vals) if vals else 0)

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(rel_labels_display, rel_means, yerr=rel_stds,
              capsize=6, color=colors, alpha=0.85, edgecolor="black")
ax.axhline(np.mean(rel_means), color="red", linestyle="--",
           label=f"Mean = {np.mean(rel_means):.3f}")
for bar, v, s in zip(bars, rel_means, rel_stds):
    ax.text(bar.get_x() + bar.get_width()/2, v + s + 0.01,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0.4, 1.0); ax.set_ylabel("Accuracy")
ax.set_title(f"Per-Relation Accuracy ± Std — {BACKBONE}")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

metrics_names = ['Accuracy', 'False Positive Rate']
metrics_values = [accuracy, false_positive_rate]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(metrics_names, metrics_values, color=['skyblue', 'salmon'])

ax.set_ylim(0, 1) # Metrics are usually between 0 and 1
ax.set_ylabel('Value')
ax.set_title('Test Metrics (Mean Across Folds)')

# Add value labels on top of the bars
for i, v in enumerate(metrics_values):
    ax.text(i, v + 0.05, f'{v:.4f}', ha='center', va='bottom')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()